In [0]:
#lis la table bronze
from pyspark.sql import functions as F

df_stops_bronze = spark.table("workspace.sncf_bronze.stops")

display(df_stops_bronze.limit(20))

stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,_ingestion_timestamp,_source_system,_source_file,_batch_id
StopArea:OCE71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETGV INOUI-71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,0,StopArea:OCE71043075,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETGV INOUI-71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,0,StopArea:OCE71718010,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE71793000,GIRONA,null,41.97937100,2.816957000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETGV INOUI-71793000,GIRONA,null,41.97937100,2.816957000,null,null,0,StopArea:OCE71793000,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE71793150,Portbou,null,42.42470100,3.158034000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCECar TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETrain TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE80021402,Augsburg Hbf,null,48.36540000,10.88600000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447


In [0]:
df_stops_bronze.printSchema()

root
 |-- stop_id: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_desc: string (nullable = true)
 |-- stop_lat: string (nullable = true)
 |-- stop_lon: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- stop_url: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- parent_station: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [0]:
df_stops_silver = (
    df_stops_bronze

    # Nettoyage texte
    .withColumn("stop_id", F.trim(F.col("stop_id")))
    .withColumn("stop_name", F.trim(F.col("stop_name")))

    # Typage
    .withColumn("stop_lat", F.col("stop_lat").cast("double"))
    .withColumn("stop_lon", F.col("stop_lon").cast("double"))
    .withColumn("location_type", F.col("location_type").cast("int"))

    # Nettoyage chaîne vide -> NULL
    .withColumn(
        "parent_station",
        F.when(
            F.trim(F.col("parent_station")) == "",
            None
        ).otherwise(F.trim(F.col("parent_station")))
    )
)

In [0]:
df_stops_silver.printSchema()

display(
    df_stops_silver.select(
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon",
        "location_type",
        "parent_station"
    ).limit(20)
)

root
 |-- stop_id: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_desc: string (nullable = true)
 |-- stop_lat: double (nullable = true)
 |-- stop_lon: double (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- stop_url: string (nullable = true)
 |-- location_type: integer (nullable = true)
 |-- parent_station: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
StopArea:OCE71043075,FIGUERES-VILAFANT,42.264581,2.943028,1,null
StopPoint:OCETGV INOUI-71043075,FIGUERES-VILAFANT,42.264581,2.943028,0,StopArea:OCE71043075
StopArea:OCE71718010,Barcelone-Sants,41.378961,2.139834,1,null
StopPoint:OCETGV INOUI-71718010,Barcelone-Sants,41.378961,2.139834,0,StopArea:OCE71718010
StopArea:OCE71793000,GIRONA,41.979371,2.816957,1,null
StopPoint:OCETGV INOUI-71793000,GIRONA,41.979371,2.816957,0,StopArea:OCE71793000
StopArea:OCE71793150,Portbou,42.424701,3.158034,1,null
StopPoint:OCECar TER-71793150,Portbou,42.424701,3.158034,0,StopArea:OCE71793150
StopPoint:OCETrain TER-71793150,Portbou,42.424701,3.158034,0,StopArea:OCE71793150
StopArea:OCE80021402,Augsburg Hbf,48.3654,10.886,1,null


In [0]:
#quality
display(
    df_stops_silver.filter(
        F.col("stop_id").isNull()
        | F.col("stop_name").isNull()
        | F.col("stop_lat").isNull()
        | F.col("stop_lon").isNull()
    )
)

stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,_ingestion_timestamp,_source_system,_source_file,_batch_id


In [0]:
display(
    df_stops_silver
    .groupBy("stop_id")
    .count()
    .filter(F.col("count") > 1)
)
total_rows = df_stops_silver.count()
distinct_ids = df_stops_silver.select("stop_id").distinct().count()

print("Nombre total de lignes :", total_rows)
print("Nombre de stop_id distincts :", distinct_ids)

stop_id,count


Nombre total de lignes : 8678
Nombre de stop_id distincts : 8678


In [0]:
#<r retirer les lignes où la clé est absente, et garder uniquement des coordonnées plausibles
df_stops_silver_clean = (
    df_stops_silver
    .filter(F.col("stop_id").isNotNull())
    .filter(
        F.col("stop_lat").between(-90, 90)
        & F.col("stop_lon").between(-180, 180)
    )
)

In [0]:
print("Avant nettoyage :", df_stops_silver.count())
print("Après nettoyage :", df_stops_silver_clean.count())

Avant nettoyage : 8678
Après nettoyage : 8678


In [0]:
df_stops_silver_final = (
    df_stops_silver_clean
    .withColumn("_silver_processed_at", F.current_timestamp())
)

In [0]:
(
    df_stops_silver_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.stops")
)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. LECTURE DE LA TABLE BRONZE
# ============================================================

df_routes_bronze = spark.table("workspace.sncf_bronze.routes")


# ============================================================
# 2. NETTOYAGE ET TYPAGE POUR LA COUCHE SILVER
# ============================================================

df_routes_silver = (
    df_routes_bronze

    # Nettoyage des colonnes texte
    .withColumn("route_id", F.trim(F.col("route_id")))
    .withColumn("agency_id", F.trim(F.col("agency_id")))
    .withColumn("route_short_name", F.trim(F.col("route_short_name")))
    .withColumn("route_long_name", F.trim(F.col("route_long_name")))
    .withColumn("route_desc", F.trim(F.col("route_desc")))

    # route_type doit être numérique
    .withColumn("route_type", F.col("route_type").cast("int"))

    # Nettoyage des codes couleurs
    # chaîne vide -> NULL
    # conversion en majuscules
    .withColumn(
        "route_color",
        F.when(
            F.trim(F.col("route_color")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("route_color"))))
    )

    .withColumn(
        "route_text_color",
        F.when(
            F.trim(F.col("route_text_color")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("route_text_color"))))
    )
)


# ============================================================
# 3. CONTRÔLES QUALITÉ
# ============================================================

print("===== CONTRÔLES ROUTES =====")

# Nombre de lignes
print("Nombre de lignes Bronze :", df_routes_bronze.count())
print("Nombre de lignes après transformation :", df_routes_silver.count())

# route_id est notre clé : elle ne doit pas être NULL
null_route_id = (
    df_routes_silver
    .filter(F.col("route_id").isNull())
    .count()
)

print("route_id NULL :", null_route_id)


# Recherche des route_id dupliqués
duplicates = (
    df_routes_silver
    .groupBy("route_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Nombre de route_id dupliqués :", duplicates.count())


# Vérification des différents types de transport
display(
    df_routes_silver
    .groupBy("route_type")
    .count()
    .orderBy("route_type")
)


# Affichage d'un échantillon pour contrôle visuel
display(
    df_routes_silver.select(
        "route_id",
        "agency_id",
        "route_short_name",
        "route_long_name",
        "route_type",
        "route_color"
    ).limit(20)
)

===== CONTRÔLES ROUTES =====
Nombre de lignes Bronze : 697
Nombre de lignes après transformation : 697
route_id NULL : 0
Nombre de route_id dupliqués : 0


route_type,count
0,4
2,579
3,114


route_id,agency_id,route_short_name,route_long_name,route_type,route_color
FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,1187,P53,Bening - Sarreguemines,3,006600
FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,1187,C30,Saint-Étienne - Roanne,2,0749FF
FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,1187,C6,Marseille - Toulon - Hyeres,2,0749FF
FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,1187,C8,Montpellier Saint-Roch - Avignon Centre,2,0749FF
FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,1187,C20,Cholet Angers Saumur,2,0749FF
FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,1187,P13,Ambérieu - Mâcon,2,006600
FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,1187,K1,Toulouse Matabiau - Brive La Gaillarde,2,BF005F
FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,1187,R16,Ussel - Bort Les Orgues,3,006600
FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,1187,L22,22. Limoges - Uzerche - Brive,2,3E844C
FR:Line::03A83359-7D9D-4301-9C87-92FED4666451:,1187,K3+,Paris - Deauville,2,6E1E78


In [0]:
# ============================================================
# 4. FINALISATION DE LA TABLE SILVER
# ============================================================

df_routes_silver_final = (
    df_routes_silver

    # Suppression des lignes sans clé route_id
    .filter(F.col("route_id").isNotNull())

    # Métadonnée permettant de savoir quand
    # la transformation Silver a été exécutée
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)


# ============================================================
# 5. ÉCRITURE DANS LA COUCHE SILVER
# ============================================================

(
    df_routes_silver_final.write
    .format("delta")              # Format Delta Lake
    .mode("overwrite")            # Remplace la table existante
    .saveAsTable(
        "workspace.sncf_silver.routes"
    )
)


# ============================================================
# 6. VALIDATION POST-ÉCRITURE
# ============================================================

df_routes_result = spark.table(
    "workspace.sncf_silver.routes"
)

print("===== RÉSULTAT INGESTION SILVER =====")

print(
    "Nombre de lignes Bronze :",
    df_routes_bronze.count()
)

print(
    "Nombre de lignes Silver :",
    df_routes_result.count()
)

print("\nSchéma Silver :")
df_routes_result.printSchema()


# Vérification visuelle finale
display(df_routes_result.limit(20))

===== RÉSULTAT INGESTION SILVER =====
Nombre de lignes Bronze : 697
Nombre de lignes Silver : 697

Schéma Silver :
root
 |-- route_id: string (nullable = true)
 |-- agency_id: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- route_desc: string (nullable = true)
 |-- route_type: integer (nullable = true)
 |-- route_url: string (nullable = true)
 |-- route_color: string (nullable = true)
 |-- route_text_color: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at
FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,1187,P53,Bening - Sarreguemines,null,3,null,006600,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,1187,C30,Saint-Étienne - Roanne,null,2,null,0749FF,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,1187,C6,Marseille - Toulon - Hyeres,null,2,null,0749FF,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,1187,C8,Montpellier Saint-Roch - Avignon Centre,null,2,null,0749FF,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,1187,C20,Cholet Angers Saumur,null,2,null,0749FF,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,1187,P13,Ambérieu - Mâcon,null,2,null,006600,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,1187,K1,Toulouse Matabiau - Brive La Gaillarde,null,2,null,BF005F,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,1187,R16,Ussel - Bort Les Orgues,null,3,null,006600,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,1187,L22,22. Limoges - Uzerche - Brive,null,2,null,3E844C,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z
FR:Line::03A83359-7D9D-4301-9C87-92FED4666451:,1187,K3+,Paris - Deauville,null,2,null,6E1E78,FFFFFF,dbfs:/Volumes/workspace/sncf_bronze/landing/routes.txt,2026-09-04T09:36:52.317Z,SNCF_GTFS,20260904093652,2026-09-04T11:39:01.682Z


In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. LECTURE BRONZE
# ============================================================

df_trips_bronze = spark.table("workspace.sncf_bronze.trips")

# ============================================================
# 2. NETTOYAGE / TYPAGE
# ============================================================

df_trips_silver = (
    df_trips_bronze
    .withColumn("route_id", F.trim(F.col("route_id")))
    .withColumn("service_id", F.trim(F.col("service_id")))
    .withColumn("trip_id", F.trim(F.col("trip_id")))
    .withColumn("trip_headsign", F.trim(F.col("trip_headsign")))
    .withColumn("direction_id", F.col("direction_id").cast("int"))
)

# ============================================================
# 3. CONTRÔLES QUALITÉ
# ============================================================

print("===== CONTRÔLES TRIPS =====")
print("Bronze :", df_trips_bronze.count())

print(
    "trip_id NULL :",
    df_trips_silver.filter(F.col("trip_id").isNull()).count()
)

duplicates = (
    df_trips_silver
    .groupBy("trip_id")
    .count()
    .filter(F.col("count") > 1)
)

print("trip_id dupliqués :", duplicates.count())

# Vérification route_id vers routes
df_routes_ref = spark.table(
    "workspace.sncf_silver.routes"
).select("route_id")

invalid_routes = (
    df_trips_silver
    .select("route_id")
    .distinct()
    .join(df_routes_ref, "route_id", "left_anti")
)

print("route_id inexistants :", invalid_routes.count())

display(df_trips_silver.limit(20))

===== CONTRÔLES TRIPS =====
Bronze : 39921
trip_id NULL : 0
trip_id dupliqués : 0
route_id inexistants : 0


route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,_source_file,_ingestion_timestamp,_source_system,_batch_id
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000001,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,105241,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000002,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,105342,1,2,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000003,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,105347,1,3,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000004,OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,105350,1,4,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000005,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,105362,1,5,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000006,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1045:20261211,105362,1,6,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000007,OCESN105375F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1021:20261009,105375,1,7,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000008,OCESN105375F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1023:20261212,105375,1,8,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000009,OCESN105377F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1327:20261212,105377,1,9,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000009,OCESN105378F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1609:20261212,105378,1,10,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657


In [0]:
# ============================================================
# 4. FINALISATION
# ============================================================

df_trips_final = (
    df_trips_silver
    .filter(F.col("trip_id").isNotNull())
    .filter(F.col("route_id").isNotNull())
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# ============================================================
# 5. ÉCRITURE SILVER
# ============================================================

(
    df_trips_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.trips")
)

# ============================================================
# 6. VALIDATION
# ============================================================

df_result = spark.table("workspace.sncf_silver.trips")

print("Bronze :", df_trips_bronze.count())
print("Silver :", df_result.count())

df_result.printSchema()
display(df_result.limit(20))

Bronze : 39921
Silver : 39921
root
 |-- route_id: string (nullable = true)
 |-- service_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- trip_headsign: string (nullable = true)
 |-- direction_id: integer (nullable = true)
 |-- block_id: string (nullable = true)
 |-- shape_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000001,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,105241,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000002,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,105342,1,2,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000003,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,105347,1,3,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000004,OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,105350,1,4,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000005,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,105362,1,5,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000006,OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1045:20261211,105362,1,6,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000007,OCESN105375F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1021:20261009,105375,1,7,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000008,OCESN105375F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1023:20261212,105375,1,8,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000009,OCESN105377F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1327:20261212,105377,1,9,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z
FR:Line::85d3579c-996b-4671-89af-839855ede78a:,000009,OCESN105378F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1609:20261212,105378,1,10,null,dbfs:/Volumes/workspace/sncf_bronze/landing/trips.txt,2026-09-04T09:36:57.823Z,SNCF_GTFS,20260904093657,2026-09-04T11:39:11.745Z


In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. LECTURE BRONZE
# ============================================================

df_stop_times_bronze = spark.table(
    "workspace.sncf_bronze.stop_times"
)

# ============================================================
# 2. NETTOYAGE / TYPAGE
# ============================================================

df_stop_times_silver = (
    df_stop_times_bronze
    .withColumn("trip_id", F.trim(F.col("trip_id")))
    .withColumn("arrival_time", F.trim(F.col("arrival_time")))
    .withColumn("departure_time", F.trim(F.col("departure_time")))
    .withColumn("stop_id", F.trim(F.col("stop_id")))
    .withColumn("stop_sequence", F.col("stop_sequence").cast("int"))
    .withColumn("pickup_type", F.col("pickup_type").cast("int"))
    .withColumn("drop_off_type", F.col("drop_off_type").cast("int"))
)

# ============================================================
# 3. CONTRÔLES QUALITÉ
# ============================================================

print("===== CONTRÔLES STOP_TIMES =====")
print("Bronze :", df_stop_times_bronze.count())

print(
    "trip_id NULL :",
    df_stop_times_silver.filter(F.col("trip_id").isNull()).count()
)

print(
    "stop_id NULL :",
    df_stop_times_silver.filter(F.col("stop_id").isNull()).count()
)

duplicates = (
    df_stop_times_silver
    .groupBy("trip_id", "stop_sequence")
    .count()
    .filter(F.col("count") > 1)
)

print("Doublons trip_id + stop_sequence :", duplicates.count())

# Intégrité vers trips
df_trips_ref = spark.table(
    "workspace.sncf_silver.trips"
).select("trip_id")

invalid_trips = (
    df_stop_times_silver
    .select("trip_id")
    .distinct()
    .join(df_trips_ref, "trip_id", "left_anti")
)

print("trip_id inexistants :", invalid_trips.count())

# Intégrité vers stops
df_stops_ref = spark.table(
    "workspace.sncf_silver.stops"
).select("stop_id")

invalid_stops = (
    df_stop_times_silver
    .select("stop_id")
    .distinct()
    .join(df_stops_ref, "stop_id", "left_anti")
)

print("stop_id inexistants :", invalid_stops.count())

display(df_stop_times_silver.limit(20))

===== CONTRÔLES STOP_TIMES =====
Bronze : 344930
trip_id NULL : 0
stop_id NULL : 0
Doublons trip_id + stop_sequence : 0
trip_id inexistants : 0
stop_id inexistants : 0


trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,_source_file,_ingestion_timestamp,_source_system,_batch_id
OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,19:54:00,19:54:00,StopPoint:OCENavette-87571000,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,19:59:00,19:59:00,StopPoint:OCENavette-87571240,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,17:23:00,17:23:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,17:28:00,17:28:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,22:10:00,22:10:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,22:15:00,22:15:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,05:55:00,05:55:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,06:00:00,06:00:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,10:40:00,10:40:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654
OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,10:46:00,10:46:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654


In [0]:
df_stop_times_final = (
    df_stop_times_silver
    .filter(F.col("trip_id").isNotNull())
    .filter(F.col("stop_id").isNotNull())
    .filter(F.col("stop_sequence").isNotNull())
    .withColumn("_silver_processed_at", F.current_timestamp())
)

(
    df_stop_times_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.stop_times")
)

df_result = spark.table("workspace.sncf_silver.stop_times")

print("Bronze :", df_stop_times_bronze.count())
print("Silver :", df_result.count())

df_result.printSchema()
display(df_result.limit(20))

Bronze : 344930
Silver : 344930
root
 |-- trip_id: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- stop_headsign: string (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- shape_dist_traveled: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at
OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,19:54:00,19:54:00,StopPoint:OCENavette-87571000,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571000:87571240:2:1959:20261030,19:59:00,19:59:00,StopPoint:OCENavette-87571240,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,17:23:00,17:23:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1728:20261212,17:28:00,17:28:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,22:10:00,22:10:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:2215:20261211,22:15:00,22:15:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,05:55:00,05:55:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105350F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:600:20261207,06:00:00,06:00:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,10:40:00,10:40:00,StopPoint:OCENavette-87571240,0,null,0,1,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z
OCESN105362F1187_F:NAV:FR:Line::85d3579c-996b-4671-89af-839855ede78a::87571240:87571000:2:1046:20261007,10:46:00,10:46:00,StopPoint:OCENavette-87571000,1,null,1,0,null,dbfs:/Volumes/workspace/sncf_bronze/landing/stop_times.txt,2026-09-04T09:36:54.589Z,SNCF_GTFS,20260904093654,2026-09-04T11:39:23.607Z


In [0]:
#callandar date 
from pyspark.sql import functions as F

df_calendar_dates_bronze = spark.table(
    "workspace.sncf_bronze.calendar_dates"
)

df_calendar_dates_silver = (
    df_calendar_dates_bronze
    .withColumn("service_id", F.trim(F.col("service_id")))
    .withColumn(
        "date",
        F.to_date(F.col("date"), "yyyyMMdd")
    )
    .withColumn(
        "exception_type",
        F.col("exception_type").cast("int")
    )
)

print("===== CONTRÔLES CALENDAR_DATES =====")

print(
    "service_id NULL :",
    df_calendar_dates_silver
    .filter(F.col("service_id").isNull())
    .count()
)

print(
    "date NULL :",
    df_calendar_dates_silver
    .filter(F.col("date").isNull())
    .count()
)

duplicates = (
    df_calendar_dates_silver
    .groupBy("service_id", "date")
    .count()
    .filter(F.col("count") > 1)
)

print("Doublons service_id + date :", duplicates.count())

display(df_calendar_dates_silver.limit(20))

===== CONTRÔLES CALENDAR_DATES =====
service_id NULL : 0
date NULL : 0
Doublons service_id + date : 0


service_id,date,exception_type,_source_file,_ingestion_timestamp,_source_system,_batch_id
000001,2026-10-19,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-20,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-21,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-22,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-23,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-26,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-27,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-28,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-29,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647
000001,2026-10-30,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647


In [0]:
df_calendar_dates_final = (
    df_calendar_dates_silver
    .filter(F.col("service_id").isNotNull())
    .filter(F.col("date").isNotNull())
    .withColumn("_silver_processed_at", F.current_timestamp())
)

(
    df_calendar_dates_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.calendar_dates")
)

df_result = spark.table(
    "workspace.sncf_silver.calendar_dates"
)

print("Bronze :", df_calendar_dates_bronze.count())
print("Silver :", df_result.count())

df_result.printSchema()
display(df_result.limit(20))

Bronze : 172028
Silver : 172028
root
 |-- service_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- exception_type: integer (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



service_id,date,exception_type,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at
000001,2026-10-19,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-20,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-21,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-22,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-23,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-26,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-27,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-28,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-29,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z
000001,2026-10-30,1,dbfs:/Volumes/workspace/sncf_bronze/landing/calendar_dates.txt,2026-09-04T09:36:47.849Z,SNCF_GTFS,20260904093647,2026-09-04T11:39:31.950Z


In [0]:
from pyspark.sql import functions as F

df_agency_bronze = spark.table(
    "workspace.sncf_bronze.agency"
)

df_agency_silver = (
    df_agency_bronze
    .withColumn("agency_id", F.trim(F.col("agency_id")))
    .withColumn("agency_name", F.trim(F.col("agency_name")))
    .withColumn("agency_url", F.trim(F.col("agency_url")))
    .withColumn("agency_timezone", F.trim(F.col("agency_timezone")))
)

print("===== CONTRÔLES AGENCY =====")

print(
    "agency_id NULL :",
    df_agency_silver.filter(F.col("agency_id").isNull()).count()
)

duplicates = (
    df_agency_silver
    .groupBy("agency_id")
    .count()
    .filter(F.col("count") > 1)
)

print("agency_id dupliqués :", duplicates.count())

display(df_agency_silver.limit(20))

===== CONTRÔLES AGENCY =====
agency_id NULL : 0
agency_id dupliqués : 0


agency_id,agency_name,agency_url,agency_timezone,agency_lang,_source_file,_ingestion_timestamp,_source_system,_batch_id
1187,SNCF VOYAGEURS,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645
5111,SNCF Voyageurs SA,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645
5235,SNCF Voyageurs EA,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645
5270,SNCF Voyageurs LO,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645
5429,TMR SA,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645
OCEdefault,OCEdefault,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645


In [0]:
df_agency_final = (
    df_agency_silver
    .filter(F.col("agency_id").isNotNull())
    .withColumn("_silver_processed_at", F.current_timestamp())
)

(
    df_agency_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.agency")
)

df_result = spark.table("workspace.sncf_silver.agency")

print("Bronze :", df_agency_bronze.count())
print("Silver :", df_result.count())

df_result.printSchema()
display(df_result.limit(20))

Bronze : 6
Silver : 6
root
 |-- agency_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_url: string (nullable = true)
 |-- agency_timezone: string (nullable = true)
 |-- agency_lang: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



agency_id,agency_name,agency_url,agency_timezone,agency_lang,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at
1187,SNCF VOYAGEURS,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645,2026-09-04T11:39:39.274Z
5111,SNCF Voyageurs SA,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645,2026-09-04T11:39:39.274Z
5235,SNCF Voyageurs EA,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645,2026-09-04T11:39:39.274Z
5270,SNCF Voyageurs LO,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645,2026-09-04T11:39:39.274Z
5429,TMR SA,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645,2026-09-04T11:39:39.274Z
OCEdefault,OCEdefault,http://www.sncf.com,Europe/Paris,fr,dbfs:/Volumes/workspace/sncf_bronze/landing/agency.txt,2026-09-04T09:36:45.617Z,SNCF_GTFS,20260904093645,2026-09-04T11:39:39.274Z


In [0]:
from pyspark.sql import functions as F

df_feed_bronze = spark.table(
    "workspace.sncf_bronze.feed_info"
)

df_feed_silver = (
    df_feed_bronze
    .withColumn(
        "feed_publisher_name",
        F.trim(F.col("feed_publisher_name"))
    )
    .withColumn(
        "feed_publisher_url",
        F.trim(F.col("feed_publisher_url"))
    )
    .withColumn(
        "feed_lang",
        F.trim(F.col("feed_lang"))
    )
    .withColumn(
        "feed_start_date",
        F.to_date(F.col("feed_start_date"), "yyyyMMdd")
    )
    .withColumn(
        "feed_end_date",
        F.to_date(F.col("feed_end_date"), "yyyyMMdd")
    )
)

print("===== CONTRÔLES FEED_INFO =====")

print(
    "publisher NULL :",
    df_feed_silver
    .filter(F.col("feed_publisher_name").isNull())
    .count()
)

display(df_feed_silver)

===== CONTRÔLES FEED_INFO =====
publisher NULL : 0


feed_id,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,conv_rev,plan_rev,_source_file,_ingestion_timestamp,_source_system,_batch_id
0,SNCF,http://www.sncf.com,fr,2026-09-03,2027-02-28,2026-09-03,1.179,1788460432,dbfs:/Volumes/workspace/sncf_bronze/landing/feed_info.txt,2026-09-04T09:36:50.182Z,SNCF_GTFS,20260904093650


In [0]:
df_feed_final = (
    df_feed_silver
    .withColumn("_silver_processed_at", F.current_timestamp())
)

(
    df_feed_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.feed_info")
)

df_result = spark.table(
    "workspace.sncf_silver.feed_info"
)

print("Bronze :", df_feed_bronze.count())
print("Silver :", df_result.count())

df_result.printSchema()
display(df_result)

Bronze : 1
Silver : 1
root
 |-- feed_id: string (nullable = true)
 |-- feed_publisher_name: string (nullable = true)
 |-- feed_publisher_url: string (nullable = true)
 |-- feed_lang: string (nullable = true)
 |-- feed_start_date: date (nullable = true)
 |-- feed_end_date: date (nullable = true)
 |-- feed_version: string (nullable = true)
 |-- conv_rev: string (nullable = true)
 |-- plan_rev: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



feed_id,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,conv_rev,plan_rev,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at
0,SNCF,http://www.sncf.com,fr,2026-09-03,2027-02-28,2026-09-03,1.179,1788460432,dbfs:/Volumes/workspace/sncf_bronze/landing/feed_info.txt,2026-09-04T09:36:50.182Z,SNCF_GTFS,20260904093650,2026-09-04T11:39:46.241Z


In [0]:
from pyspark.sql import functions as F

df_transfers_bronze = spark.table(
    "workspace.sncf_bronze.transfers"
)

df_transfers_silver = (
    df_transfers_bronze
    .withColumn(
        "from_stop_id",
        F.trim(F.col("from_stop_id"))
    )
    .withColumn(
        "to_stop_id",
        F.trim(F.col("to_stop_id"))
    )
    .withColumn(
        "transfer_type",
        F.col("transfer_type").cast("int")
    )
    .withColumn(
        "min_transfer_time",
        F.col("min_transfer_time").cast("int")
    )
)

print("===== CONTRÔLES TRANSFERS =====")

print(
    "from_stop_id NULL :",
    df_transfers_silver
    .filter(F.col("from_stop_id").isNull())
    .count()
)

print(
    "to_stop_id NULL :",
    df_transfers_silver
    .filter(F.col("to_stop_id").isNull())
    .count()
)

# Contrôle vers stops
df_stops_ref = spark.table(
    "workspace.sncf_silver.stops"
).select("stop_id")

invalid_from = (
    df_transfers_silver
    .select(F.col("from_stop_id").alias("stop_id"))
    .distinct()
    .join(df_stops_ref, "stop_id", "left_anti")
)

invalid_to = (
    df_transfers_silver
    .select(F.col("to_stop_id").alias("stop_id"))
    .distinct()
    .join(df_stops_ref, "stop_id", "left_anti")
)

print("from_stop_id inexistants :", invalid_from.count())
print("to_stop_id inexistants :", invalid_to.count())

display(df_transfers_silver.limit(20))

===== CONTRÔLES TRANSFERS =====
from_stop_id NULL : 0
to_stop_id NULL : 0
from_stop_id inexistants : 0
to_stop_id inexistants : 0


from_stop_id,to_stop_id,transfer_type,min_transfer_time,from_route_id,to_route_id,_source_file,_ingestion_timestamp,_source_system,_batch_id


In [0]:
df_transfers_final = (
    df_transfers_silver
    .filter(F.col("from_stop_id").isNotNull())
    .filter(F.col("to_stop_id").isNotNull())
    .withColumn("_silver_processed_at", F.current_timestamp())
)

(
    df_transfers_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_silver.transfers")
)

df_result = spark.table(
    "workspace.sncf_silver.transfers"
)

print("Bronze :", df_transfers_bronze.count())
print("Silver :", df_result.count())

df_result.printSchema()
display(df_result.limit(20))

Bronze : 0
Silver : 0
root
 |-- from_stop_id: string (nullable = true)
 |-- to_stop_id: string (nullable = true)
 |-- transfer_type: integer (nullable = true)
 |-- min_transfer_time: integer (nullable = true)
 |-- from_route_id: string (nullable = true)
 |-- to_route_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



from_stop_id,to_stop_id,transfer_type,min_transfer_time,from_route_id,to_route_id,_source_file,_ingestion_timestamp,_source_system,_batch_id,_silver_processed_at


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# GOLD - CRÉATION DES SURROGATE KEYS (PK) ET FOREIGN KEYS (FK)
# ============================================================


# ------------------------------------------------------------
# 1. DIM_STATION
# PK technique : station_sk
# Business Key : stop_id
# ------------------------------------------------------------

df_station = (
    spark.table("workspace.sncf_silver.stops")
    .dropDuplicates(["stop_id"])
    .withColumn(
        "station_sk",
        F.row_number().over(Window.orderBy("stop_id"))
    )
)

# station_sk = PK technique
# stop_id     = clé métier SNCF/GTFS


# ------------------------------------------------------------
# 2. DIM_AGENCY
# PK technique : agency_sk
# Business Key : agency_id
# ------------------------------------------------------------

df_agency = (
    spark.table("workspace.sncf_silver.agency")
    .dropDuplicates(["agency_id"])
    .withColumn(
        "agency_sk",
        F.row_number().over(Window.orderBy("agency_id"))
    )
)


# ------------------------------------------------------------
# 3. DIM_ROUTE
# PK technique : route_sk
# FK : agency_sk
# ------------------------------------------------------------

df_routes_source = (
    spark.table("workspace.sncf_silver.routes")
    .dropDuplicates(["route_id"])
)

df_route = (
    df_routes_source

    # Récupération de la FK agency_sk
    .join(
        df_agency.select("agency_id", "agency_sk"),
        on="agency_id",
        how="left"
    )

    # Création de la PK technique
    .withColumn(
        "route_sk",
        F.row_number().over(Window.orderBy("route_id"))
    )
)


# ------------------------------------------------------------
# 4. DIM_SERVICE
# PK technique : service_sk
# Business Key : service_id
# ------------------------------------------------------------

df_service = (
    spark.table("workspace.sncf_silver.trips")
    .select("service_id")
    .filter(F.col("service_id").isNotNull())
    .distinct()
    .withColumn(
        "service_sk",
        F.row_number().over(Window.orderBy("service_id"))
    )
)


# ------------------------------------------------------------
# 5. DIM_DATE
# PK : date_sk
# ------------------------------------------------------------

df_date = (
    spark.table("workspace.sncf_silver.calendar_dates")

    .select("date")
    .filter(F.col("date").isNotNull())
    .distinct()

    # Exemple : 2026-09-04 -> 20260904
    .withColumn(
        "date_sk",
        F.date_format("date", "yyyyMMdd").cast("int")
    )

    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
)


# ============================================================
# AFFICHAGE DES PK / FK CRÉÉES
# ============================================================

print("===== DIM_STATION =====")
display(
    df_station.select(
        "station_sk",
        "stop_id",
        "stop_name"
    ).limit(10)
)

print("===== DIM_AGENCY =====")
display(
    df_agency.select(
        "agency_sk",
        "agency_id",
        "agency_name"
    ).limit(10)
)

print("===== DIM_ROUTE =====")
display(
    df_route.select(
        "route_sk",
        "route_id",
        "agency_sk"
    ).limit(10)
)

print("===== DIM_SERVICE =====")
display(
    df_service.select(
        "service_sk",
        "service_id"
    ).limit(10)
)

print("===== DIM_DATE =====")
display(
    df_date.select(
        "date_sk",
        "date"
    ).limit(10)
)

===== DIM_STATION =====


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


station_sk,stop_id,stop_name
1,StopArea:OCE71043075,FIGUERES-VILAFANT
2,StopArea:OCE71718010,Barcelone-Sants
3,StopArea:OCE71793000,GIRONA
4,StopArea:OCE71793150,Portbou
5,StopArea:OCE80021402,Augsburg Hbf
6,StopArea:OCE80110684,Francfort sur le Main
7,StopArea:OCE80140087,Mannheim Hbf
8,StopArea:OCE80140210,Heidelberg Hbf
9,StopArea:OCE80142281,Karlsruhe Hbf
10,StopArea:OCE80142778,Baden-Baden


===== DIM_AGENCY =====


agency_sk,agency_id,agency_name
1,1187,SNCF VOYAGEURS
2,5111,SNCF Voyageurs SA
3,5235,SNCF Voyageurs EA
4,5270,SNCF Voyageurs LO
5,5429,TMR SA
6,OCEdefault,OCEdefault


...

route_sk,route_id,agency_sk
1,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,1
2,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,1
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,1
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,1
5,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,1
6,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,1
7,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,1
8,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,1
9,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,1
10,FR:Line::03A83359-7D9D-4301-9C87-92FED4666451:,1


===== DIM_SERVICE =====


service_sk,service_id
1,000001
2,000002
3,000003
4,000004
5,000005
6,000006
7,000007
8,000008
9,000009
10,000010


===== DIM_DATE =====


date_sk,date
20261019,2026-10-19
20261020,2026-10-20
20261021,2026-10-21
20261022,2026-10-22
20261023,2026-10-23
20261026,2026-10-26
20261027,2026-10-27
20261028,2026-10-28
20261029,2026-10-29
20261030,2026-10-30


In [0]:
from pyspark.sql import functions as F

print("====================================================")
print("       CONTROLES DATA QUALITY - GOLD KEYS")
print("====================================================")


# ============================================================
# 1. CONTROLE PK : DIM_STATION
# ============================================================

print("\n===== DIM_STATION =====")

print(
    "station_sk NULL :",
    df_station.filter(F.col("station_sk").isNull()).count()
)

print(
    "station_sk dupliqués :",
    df_station
    .groupBy("station_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "stop_id dupliqués :",
    df_station
    .groupBy("stop_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# ============================================================
# 2. CONTROLE PK : DIM_AGENCY
# ============================================================

print("\n===== DIM_AGENCY =====")

print(
    "agency_sk NULL :",
    df_agency.filter(F.col("agency_sk").isNull()).count()
)

print(
    "agency_sk dupliqués :",
    df_agency
    .groupBy("agency_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "agency_id dupliqués :",
    df_agency
    .groupBy("agency_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# ============================================================
# 3. CONTROLE PK + FK : DIM_ROUTE
# ============================================================

print("\n===== DIM_ROUTE =====")

print(
    "route_sk NULL :",
    df_route.filter(F.col("route_sk").isNull()).count()
)

print(
    "route_sk dupliqués :",
    df_route
    .groupBy("route_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "route_id dupliqués :",
    df_route
    .groupBy("route_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "agency_sk FK NULL :",
    df_route
    .filter(F.col("agency_sk").isNull())
    .count()
)


# ============================================================
# 4. CONTROLE PK : DIM_SERVICE
# ============================================================

print("\n===== DIM_SERVICE =====")

print(
    "service_sk NULL :",
    df_service.filter(F.col("service_sk").isNull()).count()
)

print(
    "service_sk dupliqués :",
    df_service
    .groupBy("service_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "service_id dupliqués :",
    df_service
    .groupBy("service_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# ============================================================
# 5. CONTROLE PK : DIM_DATE
# ============================================================

print("\n===== DIM_DATE =====")

print(
    "date_sk NULL :",
    df_date.filter(F.col("date_sk").isNull()).count()
)

print(
    "date_sk dupliqués :",
    df_date
    .groupBy("date_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "date dupliquées :",
    df_date
    .groupBy("date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# ============================================================
# 6. RESUME
# ============================================================

print("\n====================================================")
print("Si tous les NULL et doublons importants = 0")
print("alors les dimensions sont prêtes pour Gold.")
print("====================================================")

       CONTROLES DATA QUALITY - GOLD KEYS

===== DIM_STATION =====


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


station_sk NULL : 0
station_sk dupliqués : 0
stop_id dupliqués : 0

===== DIM_AGENCY =====
agency_sk NULL : 0
agency_sk dupliqués : 0
agency_id dupliqués : 0

===== DIM_ROUTE =====
route_sk NULL : 0
route_sk dupliqués : 0
route_id dupliqués : 0
agency_sk FK NULL : 0

===== DIM_SERVICE =====
service_sk NULL : 0
service_sk dupliqués : 0
service_id dupliqués : 0

===== DIM_DATE =====
date_sk NULL : 0
date_sk dupliqués : 0
date dupliquées : 0

Si tous les NULL et doublons importants = 0
alors les dimensions sont prêtes pour Gold.
